# 12 · 模型保存、加载、部署与性能优化

> 训练好的模型要能存下来、能复现、能上线。
> 这一节讲工程落地的关键技能。

## 12.1 保存与加载：三种方式

| 方式 | 保存内容 | 推荐度 |
|---|---|---|
| `state_dict` | 只存参数 | ✅ 官方推荐 |
| 整个模型 `torch.save(model)` | 用 pickle 存结构+参数 | ⚠️ 不推荐（耦合代码） |
| checkpoint | 参数 + 优化器 + epoch + 指标 | ✅ 断点续训必备 |

### 推荐：保存 state_dict

In [ ]:
import torch
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
    def forward(self, x):
        return self.net(x)

model = Net()
x = torch.randn(4, 10)

# 保存
torch.save(model.state_dict(), "model_state.pth")
print("已保存 state_dict")

# 加载：必须先创建同结构模型
model2 = Net()
state = torch.load("model_state.pth", map_location="cpu", weights_only=True)
model2.load_state_dict(state)
model2.eval()
print("加载成功，输出一致:", torch.allclose(model.eval()(x), model2(x)))

### 完整 checkpoint（断点续训）

包含：模型参数、优化器状态、epoch、最佳指标、scheduler、scaler。

In [ ]:
model = Net()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10)
epoch = 5
best_acc = 0.93

checkpoint = {
    "epoch": epoch,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),
    "best_acc": best_acc,
    "config": {"hidden": 32, "classes": 3},
}
torch.save(checkpoint, "checkpoint.pth")
print("checkpoint 已保存，字段:", list(checkpoint.keys()))

# 恢复
ckpt = torch.load("checkpoint.pth", map_location="cpu", weights_only=False)
model_r = Net()
optimizer_r = torch.optim.AdamW(model_r.parameters(), lr=1e-3)
scheduler_r = torch.optim.lr_scheduler.StepLR(optimizer_r, step_size=10)

model_r.load_state_dict(ckpt["model_state"])
optimizer_r.load_state_dict(ckpt["optimizer_state"])
scheduler_r.load_state_dict(ckpt["scheduler_state"])
print("从 epoch", ckpt["epoch"], "恢复，best_acc =", ckpt["best_acc"])

### 通用保存/加载函数（推荐模板）

```python
def save_checkpoint(path, model, optimizer=None, epoch=0, **extra):
    state = {"model": model.state_dict(), "epoch": epoch}
    if optimizer is not None:
        state["optimizer"] = optimizer.state_dict()
    state.update(extra)
    torch.save(state, path)

def load_checkpoint(path, model, optimizer=None, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    return ckpt
```

## 12.2 加载部分权重 / 参数不匹配

常见场景：换了分类头、改了层名、加载别人的权重。

In [ ]:
class Net2(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, num_classes))
    def forward(self, x):
        return self.net(x)

# 严格加载会报错（最后一层形状不同）
big = Net2(num_classes=5)
sd = model.state_dict()

try:
    big.load_state_dict(sd)
except RuntimeError as e:
    print("严格加载报错示例:\n", str(e)[:300])

In [ ]:
# 方式 1：strict=False —— 处理"层数不同"（key 有增删）
# 注意：strict=False 只能忽略"缺失/多余"的 key，形状不一致仍会报错！
class NetPlus(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
        self.extra = nn.Linear(3, 3)          # 新版多出来的一个头
    def forward(self, x):
        return self.extra(self.net(x))

big3 = NetPlus()
missing, unexpected = big3.load_state_dict(sd, strict=False)
print("缺失的 key (新模型有、ckpt 没有):", missing)
print("多余的 key (ckpt 有、新模型没有):", unexpected)

In [ ]:
# 方式 2：手动过滤 —— 处理"形状不匹配"（strict=False 对形状不一致无效）
big2 = Net2(num_classes=5)
new_sd = big2.state_dict()
filtered = {k: v for k, v in sd.items()
            if k in new_sd and v.shape == new_sd[k].shape}
new_sd.update(filtered)
big2.load_state_dict(new_sd)
print("手动过滤后加载了:", len(filtered), "个张量（net.0.* 匹配，net.2.* 形状不同被跳过）")

# 方式 3：去掉 key 前缀（如 DataParallel 的 module.）
sd_nodp = {k.replace("module.", ""): v for k, v in sd.items()}
print("前缀处理示例:", list(sd_nodp.keys())[:3])

## 12.3 多 GPU：DataParallel 与 DDP

| 方式 | 说明 | 推荐 |
|---|---|---|
| `DataParallel` (DP) | 单进程多线程，简单但慢、有 GIL 瓶颈 | 快速试 |
| `DistributedDataParallel` (DDP) | 多进程，性能好，官方推荐 | 生产 |

In [ ]:
# DataParallel 用法（有多张 GPU 时）
if torch.cuda.device_count() >= 2:
    model = Net().cuda()
    model = nn.DataParallel(model)
    print("已用 DataParallel 包装，GPU 数:", torch.cuda.device_count())
else:
    print("当前可用 GPU 数:", torch.cuda.device_count(), "（DP/DDP 演示跳过）")

In [ ]:
# DDP 训练脚本模板（需要在多进程/多机下运行 torchrun）
DDP_TEMPLATE = '''
import os, torch, torch.distributed as dist, torch.nn as nn
from torch.nn.parallel import DistributedDataParallel as DDP

def main():
    dist.init_process_group("nccl")                      # nccl for GPU
    rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(rank)

    model = Net().cuda(rank)
    model = DDP(model, device_ids=[rank])

    loader = ...          # 需配合 DistributedSampler 分片
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    for x, y in loader:
        x, y = x.cuda(rank), y.cuda(rank)
        opt.zero_grad()
        loss = nn.CrossEntropyLoss()(model(x), y)
        loss.backward()
        opt.step()
    dist.destroy_process_group()

if __name__ == "__main__":
    main()
'''
print("DDP 启动命令：")
print("  torchrun --nproc_per_node=4 train_ddp.py")
print("\n关键点：DistributedSampler、每进程独立 device、init/destroy_process_group")

> **重要**：DDP 下保存模型要存 `model.module.state_dict()`，
> 或直接用 `model.state_dict()`（DDP 会给 key 加 `module.` 前缀，加载时注意）。

## 12.4 TorchScript：脱离 Python 运行

把模型导出成可被 C++ / 移动端加载的格式。

- **trace**：记录一次前向的执行路径（要求无数据依赖的控制流）
- **script**：编译 Python 源码（支持控制流）

In [ ]:
model = Net().eval()
example = torch.randn(1, 10)

# trace
traced = torch.jit.trace(model, example)
traced.save("model_traced.pt")
print("trace 成功:", traced(example).shape)

# 加载
loaded = torch.jit.load("model_traced.pt")
print("加载后输出:", loaded(example).shape)

# script（对含 if/for 的模型更稳）
scripted = torch.jit.script(model)
print("script 成功:", scripted(example).shape)

### 动态量化（CPU 推理加速）

In [ ]:
# 动态量化：把 Linear 权重从 float32 转 int8，CPU 上提速
qmodel = torch.quantization.quantize_dynamic(Net(), {nn.Linear}, dtype=torch.qint8)
print("量化后模型结构:")
print(qmodel)
print("\n量化输出:", qmodel(example).shape)

import os
print("\n原始模型文件大小:", os.path.getsize("model_state.pth"), "bytes")
torch.save(qmodel.state_dict(), "model_quant.pth")
print("量化模型文件大小:", os.path.getsize("model_quant.pth"), "bytes")

## 12.5 ONNX 导出（跨框架部署）

ONNX 是通用模型交换格式，可被 TensorRT、OpenVINO、onnxruntime 等使用。

In [ ]:
try:
    onnx_path = "model.onnx"
    torch.onnx.export(
        model, example, onnx_path,
        input_names=["input"], output_names=["output"],
        dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
        opset_version=17,
    )
    print("ONNX 导出成功:", onnx_path)

    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    import numpy as np
    out = sess.run(None, {"input": example.numpy()})[0]
    print("onnxruntime 推理输出:", out.shape)
except ImportError as e:
    print("需要安装：pip install onnx onnxruntime")
except Exception as e:
    print("导出失败:", type(e).__name__, e)

## 12.6 torch.compile（PyTorch 2.x 加速）

一行代码获得图编译优化（TorchInductor），典型加速 1.3–2x。

In [ ]:
import time

model = Net()
x = torch.randn(256, 10)

def bench(fn, n=50, warmup=5):
    for _ in range(warmup):
        fn(x)
    t0 = time.time()
    for _ in range(n):
        fn(x)
    return time.time() - t0

t_eager = bench(model)
print(f"eager   : {t_eager:.4f}s")

try:
    compiled = torch.compile(model)
    t_compile = bench(compiled)
    print(f"compiled: {t_compile:.4f}s")
    print(f"加速比  : {t_eager / t_compile:.2f}x")
except Exception as e:
    print("torch.compile 在当前环境不可用，已跳过：", type(e).__name__, str(e)[:120])
    print("（Windows 中文环境可能因 triton 模板编码问题失败，属正常现象）")

> 首次运行 `torch.compile` 有编译开销，适合长期运行/大模型。
> 若环境不支持会退回 eager 或抛异常，用 try/except 包住即可。

## 12.7 性能分析（Profiler）

找出训练瓶颈：是数据加载慢，还是 GPU 计算慢？

In [ ]:
from torch.profiler import profile, ProfilerActivity

model = Net()
x = torch.randn(64, 10)
opt = torch.optim.Adam(model.parameters())

with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    for _ in range(5):
        opt.zero_grad()
        loss = model(x).sum()
        loss.backward()
        opt.step()

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=8))

## 12.8 推理优化清单

| 手段 | 说明 |
|---|---|
| `model.eval()` + `torch.no_grad()` | 关闭 dropout/BN 更新、不建图 |
| `torch.inference_mode()` | 比 no_grad 更快（不记录版本） |
| 半精度 / 量化 | int8 / fp16 推理 |
| `torch.compile` | 图优化 |
| ONNX + TensorRT | GPU 生产部署首选 |
| 批处理请求 | 提高吞吐 |
| 固定输入尺寸 | 启用 cudnn benchmark |
| `channels_last` | CNN 内存格式 |

In [ ]:
# inference_mode 比 no_grad 更快
import time

model = Net().eval()
x = torch.randn(512, 10)

for mode, ctx in [("no_grad", torch.no_grad()), ("inference_mode", torch.inference_mode())]:
    with ctx:
        for _ in range(5):
            model(x)
        t0 = time.time()
        for _ in range(200):
            model(x)
        print(f"{mode:16s}: {time.time()-t0:.4f}s")

## 12.9 复现性（Reproducibility）

```python
import torch, numpy as np, random
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
```
注意：完全确定性会牺牲一点性能。

## 12.10 项目结构模板

```
project/
├── data/                 # 数据
├── src/
│   ├── dataset.py        # Dataset / transforms
│   ├── model.py          # 网络定义
│   ├── train.py          # 训练脚本
│   ├── evaluate.py       # 评估
│   └── config.py         # 超参数
├── checkpoints/          # 保存的权重
├── requirements.txt
└── README.md
```

## 12.11 练习

**练习 1**：实现 `save_checkpoint` / `load_checkpoint` 函数，做一次"训练 2 epoch →
  中断 → 恢复 → 继续训练"，验证能无缝续训。

**练习 2**：把 `Net` 导出为 TorchScript，并用 `torch.jit.load` 推理。

**练习 3**：测量 `torch.compile` 在你的机器上的加速比。

**练习 4**：对比 fp32 与量化模型的文件大小和推理速度。

**练习 5**：用 profiler 找出一个含 DataLoader 的训练循环的瓶颈。

In [ ]:
# 练习 1
def save_checkpoint(path, model, optimizer=None, epoch=0, **extra):
    state = {"model": model.state_dict(), "epoch": epoch}
    if optimizer is not None:
        state["optimizer"] = optimizer.state_dict()
    state.update(extra)
    torch.save(state, path)

def load_checkpoint(path, model, optimizer=None, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    return ckpt

m = Net()
o = torch.optim.Adam(m.parameters())
save_checkpoint("ckpt_demo.pth", m, o, epoch=2, best_acc=0.9)

m2 = Net()
o2 = torch.optim.Adam(m2.parameters())
ck = load_checkpoint("ckpt_demo.pth", m2, o2)
print("练习1 恢复 epoch =", ck["epoch"], " best_acc =", ck["best_acc"])

## 12.12 小结

- 保存/加载模型用 `state_dict`；断点续训存完整 checkpoint。
- 参数不匹配：`strict=False` 或手动过滤 key。
- 部署：TorchScript（trace/script）、ONNX、量化。
- 加速：`torch.compile`、AMP、`inference_mode`、量化、ONNX+TensorRT。
- 用 profiler 定位瓶颈；固定随机种子保证复现。

## 12.13 课程总结

恭喜完成全部 13 个 notebook！你现在掌握了：

| 能力 | 对应章节 |
|---|---|
| 张量操作、自动求导 | 01–02 |
| 建模、数据管道、训练循环 | 03–05 |
| 全连接 / CNN / RNN / Transformer 实战 | 06–09 |
| 迁移学习 | 10 |
| 理论、初始化、正则、优化 | 11 |
| 保存、部署、性能 | 12 |

### 下一步建议
1. **刷数据集**：CIFAR-10 刷到 95%+，ImageNet 复现 ResNet。
2. **读论文 + 复现**：AlexNet → VGG → ResNet → Transformer → ViT → GPT。
3. **做项目**：目标检测（YOLO/Faster R-CNN）、分割（U-Net）、NLP（BERT 微调）。
4. **学框架**：PyTorch Lightning、HuggingFace、onnxruntime、TensorRT。
5. **读源码**：从 `torch/nn/modules` 开始，理解实现细节。